In [1]:
import pandas as pd

home_team= pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\home_team.csv")  # مسیر هر فایلی که میخوای چک کنی

print("shape: ", home_team.shape)
print("\n--- data type ---")
print(home_team.dtypes)
print("\n--- null ---")
print(home_team.isnull().sum())
print("\n--- duplicate ---")
print(home_team.duplicated().sum())
print("\n---  first five  ---")
print(home_team.head())
print("\n--- ---")
print(home_team.describe())

shape:  (25610, 18)

--- data type ---
match_id           int64
name                 str
slug                 str
gender               str
user_count         int64
residence            str
birthplace           str
height           float64
weight           float64
plays                str
turned_pro       float64
current_prize    float64
total_prize      float64
player_id          int64
current_rank     float64
name_code            str
country              str
full_name            str
dtype: object

--- null ---
match_id             0
name                 0
slug                 0
gender              35
user_count           0
residence        18423
birthplace       10836
height           11287
weight           18578
plays            13126
turned_pro       20806
current_prize      130
total_prize         69
player_id            0
current_rank       273
name_code            0
country              9
full_name            0
dtype: int64

--- duplicate ---
7710

---  first five  ---
   match_i

In [2]:
away_team= pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\away_team.csv")  # مسیر هر فایلی که میخوای چک کنی

print("shape:", away_team.shape)
print("\n--- data type ---")
print(away_team.dtypes)
print("\n--- null ---")
print(away_team.isnull().sum())
print("\n--- duplicate ---")
print(away_team.duplicated().sum())
print("\n--- first five---")
print(away_team.head())
print("\n------")
print(away_team.describe())

shape: (24203, 18)

--- data type ---
match_id           int64
name                 str
slug                 str
gender               str
user_count         int64
residence            str
birthplace           str
height           float64
weight           float64
plays                str
turned_pro       float64
current_prize    float64
total_prize      float64
player_id          int64
current_rank     float64
name_code            str
country              str
full_name            str
dtype: object

--- null ---
match_id             0
name                 0
slug                 0
gender              33
user_count           0
residence        17185
birthplace       10308
height           10665
weight           17293
plays            12244
turned_pro       19476
current_prize      226
total_prize        133
player_id            0
current_rank       325
name_code            0
country              5
full_name            0
dtype: int64

--- duplicate ---
7322

--- first five---
   match_id   

**QUESTION ONE**

In [3]:
# فقط دو ستون مرتبط رو انتخاب می‌کنیم
all_players = pd.concat([home_team[['player_id', 'full_name']],
                         away_team[['player_id', 'full_name']]
], ignore_index=True)

# حالا drop_duplicates بر اساس player_id (نه کل ستون‌ها)
unique_players = all_players.drop_duplicates(subset='player_id')

print("Total number of players (unique):", unique_players['player_id'].nunique())

Total number of players (unique): 2644


**QUESTION TWO**

In [4]:
all_players = pd.concat([
    home_team[['player_id', 'full_name', 'height']],
    away_team[['player_id', 'full_name', 'height']]
], ignore_index=True)

unique_players = all_players.drop_duplicates(subset='player_id')

print("Total number of players (unique):", len(unique_players))
print("null:", unique_players['height'].isnull().sum())
print("Zeros:", (unique_players['height'] == 0).sum())
print()
#print("--- cleaned data ---")
#print(sorted(unique_players['height'].dropna().unique())[:30])   # 30 تای کوچیک‌تر
#print(sorted(unique_players['height'].dropna().unique())[-30:])  # 30 تای بزرگ‌تر
#print()
print(unique_players['height'].describe())

Total number of players (unique): 2644
null: 1327
Zeros: 0

count    1317.000000
mean        1.821374
std         0.080626
min         1.570000
25%         1.780000
50%         1.830000
75%         1.880000
max         2.080000
Name: height, dtype: float64


In [5]:
total_players = len(unique_players)
missing_height = unique_players['height'].isnull().sum()
valid_height_count = unique_players['height'].notnull().sum()

avg_height = unique_players['height'].mean()

print(f"total players: {total_players}")
print(f"playes with missing height values(null): {missing_height} ({missing_height/total_players*100:.1f}%)")
print(f"players with height values: {valid_height_count}")
print(f"average height(M): {avg_height:.2f} for {valid_height_count} players")

total players: 2644
playes with missing height values(null): 1327 (50.2%)
players with height values: 1317
average height(M): 1.82 for 1317 players


**QUESTION THREE**

In [6]:
event_dfr = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\event.csv")
#CLEAN DATA FRAME AND DROPDUPLICATES
home_df = home_team.drop_duplicates().drop_duplicates(subset='match_id', keep='first')
away_df = away_team.drop_duplicates().drop_duplicates(subset='match_id', keep='first')

#KEEP THE COLS WE NEED
event_clean = event_dfr[['match_id', 'winner_code']].dropna(subset=['winner_code'])
home_clean = home_df[['match_id', 'player_id', 'full_name' , "country"]]
away_clean = away_df[['match_id', 'player_id', 'full_name',"country"]]

In [7]:
merged = event_clean.merge(home_clean, on='match_id', how='inner')
merged = merged.merge(away_clean, on='match_id', how='inner', suffixes=('_home', '_away'))

print("shape afte rmerged:", merged.shape)


shape afte rmerged: (19141, 8)


In [8]:
import numpy as np
#CHECK IF THE PLAYERS ARE HOME OR AWAY
merged['winner_id'] = np.where(merged['winner_code'] == 1, merged['player_id_home'], merged['player_id_away'])
merged['winner_name'] = np.where(merged['winner_code'] == 1, merged['full_name_home'], merged['full_name_away'])

#WIN COUNTS FOR REACH PLAYER
win_counts = merged.groupby(['winner_id', 'winner_name']).size().reset_index(name='wins')
win_counts = win_counts.sort_values('wins', ascending=False)

print("\n--- top ten players with the most win---")
print(win_counts.head(10))

top_player = win_counts.iloc[0]
print(f"\n final answer: {top_player['winner_name']} with {int(top_player['wins'])} have the highest win.")


--- top ten players with the most win---
      winner_id                        winner_name  wins
175       50901                      Popko, Dmitry    57
1131     231620                   Chidekh, Clement    53
1684     341818                       Faria, Jaime    44
366       82133  Dellien Velasco, Murkel Alejandro    42
1116     230049              Jianu, Filip Cristian    39
857      197809                        Clarke, Jay    39
906      202572                      Gengel, Marek    38
241       58515                  Collins, Danielle    38
13        16683                 Kukushkin, Mikhail    37
208       53483                      Masur, Daniel    36

 final answer: Popko, Dmitry with 57 have the highest win.


In [9]:
win_counts.head(10)

,winner_id,winner_name,wins
175,50901,"Popko, Dmitry",57
1131,231620,"Chidekh, Clement",53
1684,341818,"Faria, Jaime",44
366,82133,"Dellien Velasco, Murkel Alejandro",42
1116,230049,"Jianu, Filip Cristian",39
857,197809,"Clarke, Jay",39
906,202572,"Gengel, Marek",38
241,58515,"Collins, Danielle",38
13,16683,"Kukushkin, Mikhail",37
208,53483,"Masur, Daniel",36


In [24]:
# لیست match_id هایی که برنده‌شون top_player_id بوده رو دربیار
top_player = merged[merged['winner_id'] == top_player]['match_id'].tolist()
print(len(top_player_matches))
print(sorted(top_player_matches))

KeyError: 'winner_id'

**QUESTION FOUR**

In [10]:
time_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\time.csv")

print("time.shape:", time_df.shape)
print("\n cls", list(time_df.columns))
print("\n--- first five---")
print(time_df.head())

time.shape: (35671, 7)

 cls ['match_id', 'period_1', 'period_2', 'period_3', 'period_4', 'period_5', 'current_period_start_timestamp']

--- first five---
   match_id  period_1  period_2  period_3  period_4  period_5  \
0  11974053       NaN       NaN       NaN       NaN       NaN   
1  11974066       NaN       NaN       NaN       NaN       NaN   
2  11998445    3259.0    2639.0    4202.0       NaN       NaN   
3  11998446    2488.0    2375.0       NaN       NaN       NaN   
4  11998447    3741.0    1913.0       NaN       NaN       NaN   

   current_period_start_timestamp  
0                             NaN  
1                             NaN  
2                    1.706817e+09  
3                    1.706804e+09  
4                    1.706798e+09  


In [11]:
time_df = time_df.drop_duplicates().drop_duplicates(subset='match_id', keep='first')

period_cols = ['period_1', 'period_2', 'period_3', 'period_4', 'period_5']
time_df['total_seconds'] = time_df[period_cols].sum(axis=1)
time_df['total_hours'] = time_df['total_seconds'] / 3600

print("time.csv cleaned matches:", len(time_df))

time.csv cleaned matches: 16873


In [12]:
home_score = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis/home_team_score.csv")
away_score = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis/away_team_score.csv")

home_score = home_score.drop_duplicates().drop_duplicates(subset='match_id', keep='first')
away_score = away_score.drop_duplicates().drop_duplicates(subset='match_id', keep='first')

home_score['home_games'] = home_score[period_cols].sum(axis=1)
away_score['away_games'] = away_score[period_cols].sum(axis=1)

games = home_score[['match_id', 'home_games']].merge(
    away_score[['match_id', 'away_games']], on='match_id', how='inner'
)
games['total_games'] = games['home_games'] + games['away_games']

print("matches with game counts:", len(games))

matches with game counts: 16873


In [13]:
merged = time_df.merge(games[['match_id', 'total_games']], on='match_id', how='inner')
merged = merged[merged['total_games'] > 0]

merged['min_per_game'] = (merged['total_seconds'] / merged['total_games']) / 60

print(merged['min_per_game'].describe())

count    14587.000000
mean         3.748252
std          6.615034
min          0.000000
25%          0.000000
50%          4.316092
75%          5.042604
max        330.186275
Name: min_per_game, dtype: float64


In [14]:
# realistic range: 1 to 10 minutes per game
clean = merged[(merged['min_per_game'] >= 1) & (merged['min_per_game'] <= 10)].copy()

print("dropped as unrealistic:", len(merged) - len(clean))
print("final clean matches:", len(clean))
print(clean['total_hours'].describe())

dropped as unrealistic: 4400
final clean matches: 10187
count    10187.000000
mean         1.711743
std          0.583242
min          0.328333
25%          1.283333
50%          1.604722
75%          2.070417
max          5.256944
Name: total_hours, dtype: float64


In [15]:
longest = clean.sort_values('total_seconds', ascending=False).iloc[0]

print("longest match row:")
print(longest[['match_id', 'total_seconds', 'total_hours', 'total_games', 'min_per_game']])

longest match row:
match_id         1.217063e+07
total_seconds    1.892500e+04
total_hours      5.256944e+00
total_games      4.000000e+01
min_per_game     7.885417e+00
Name: 13433, dtype: float64


In [16]:

target_id = longest['match_id']
home_name = home_team.loc[home_team['match_id'] == target_id, 'full_name'].values
away_name = away_team.loc[away_team['match_id'] == target_id, 'full_name'].values

print(f"\nfinal answer: longest match = {target_id}, between {home_name[0]} and {away_name[0]}, duration = {longest['total_hours']:.2f} hours, {int(longest['total_games'])} games played ({longest['min_per_game']:.2f} min/game)")


final answer: longest match = 12170629.0, between Kung, Leonie and Scilipoti, Sebastianna, duration = 5.26 hours, 40 games played (7.89 min/game)


**QUESTION FIVE**

In [17]:
import pandas as pd

home_score = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis/home_team_score.csv")

print("shape: ", home_score.shape)
print("\n--- data type ---")
print(home_score.dtypes)
print("\n--- null ---")
print(home_score.isnull().sum())
print("\n--- duplicate ---")
print(home_score.duplicated().sum())
print("\n---  first five  ---")
print(home_score.head())

shape:  (35164, 14)

--- data type ---
match_id                int64
current_score         float64
display_score         float64
period_1              float64
period_2              float64
period_3              float64
period_4              float64
period_5              float64
period_1_tie_break    float64
period_2_tie_break    float64
period_3_tie_break    float64
period_4_tie_break    float64
period_5_tie_break    float64
normal_time           float64
dtype: object

--- null ---
match_id                  0
current_score          3014
display_score          3014
period_1               3026
period_2               3357
period_3              26322
period_4              35164
period_5              35164
period_1_tie_break    32610
period_2_tie_break    32771
period_3_tie_break    34513
period_4_tie_break    35164
period_5_tie_break    35164
normal_time           35164
dtype: int64

--- duplicate ---
13618

---  first five  ---
   match_id  current_score  display_score  period_1  period_2

In [18]:
away_score = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis/away_team_score.csv")

print("shape:", away_score.shape)
print("\n--- data type ---")
print(away_score.dtypes)
print("\n--- null ---")
print(away_score.isnull().sum())
print("\n--- duplicate ---")
print(away_score.duplicated().sum())
print("\n--- first five---")
print(away_score.head())

shape: (35053, 14)

--- data type ---
match_id                int64
current_score         float64
display_score         float64
period_1              float64
period_2              float64
period_3              float64
period_4              float64
period_5              float64
period_1_tie_break    float64
period_2_tie_break    float64
period_3_tie_break    float64
period_4_tie_break    float64
period_5_tie_break    float64
normal_time           float64
dtype: object

--- null ---
match_id                  0
current_score          3013
display_score          3013
period_1               3026
period_2               3346
period_3              26246
period_4              35053
period_5              35053
period_1_tie_break    32509
period_2_tie_break    32668
period_3_tie_break    34403
period_4_tie_break    35053
period_5_tie_break    35053
normal_time           35053
dtype: int64

--- duplicate ---
13518

--- first five---
   match_id  current_score  display_score  period_1  period_2  pe

In [19]:
# CLEAN DATA FRAMES AND DROP DUPLICATES
home_df = home_score.drop_duplicates().drop_duplicates(subset='match_id',)
away_df = away_score.drop_duplicates().drop_duplicates(subset='match_id', )

print("home shape after cleaning:", home_df.shape)
print("away shape after cleaning:", away_df.shape)

home shape after cleaning: (16873, 14)
away shape after cleaning: (16873, 14)


In [20]:
period_cols = ['period_1', 'period_2', 'period_3', 'period_4', 'period_5']

home_df['sets_home'] = home_df[period_cols].notna().sum(axis=1)
away_df['sets_away'] = away_df[period_cols].notna().sum(axis=1)

merged = home_df[['match_id', 'sets_home']].merge(
    away_df[['match_id', 'sets_away']], on='match_id', how='inner')

merged['sets_played'] = merged[['sets_home', 'sets_away']].max(axis=1)

print("shape after merged:", merged.shape)
print(merged.head())

shape after merged: (16873, 4)
   match_id  sets_home  sets_away  sets_played
0  11974053          0          0            0
1  11974066          0          0            0
2  11998445          3          3            3
3  11998446          2          2            2
4  11998447          2          2            2


In [ ]:
# DROP INCOMPLETE MATCHES (0 or 1 sets = walkover / bad data)
valid = merged[merged['sets_played'] >= 2]

print("total valid matches:", len(valid))
print("\n--- sets played distribution ---")
print(valid['sets_played'].value_counts().sort_index())

avg_sets = valid['sets_played'].mean()
median_sets = valid['sets_played'].median()
mode_sets = valid['sets_played'].mode()[0]

print(f"\nfinal answer: average sets played = {avg_sets:.2f}, median = {median_sets}, most common = {mode_sets} sets")

total valid matches: 14395

--- sets played distribution ---
sets_played
2    10397
3     3998
Name: count, dtype: int64

final answer: average sets played = 2.28, median = 2.0, most common = 2 sets


**QUESTION SIX**

In [22]:
import pandas as pd
players_country = pd.concat([home_df[['player_id', 'full_name', 'country']],
    away_df[['player_id', 'full_name', 'country']]
], ignore_index=True).drop_duplicates(subset='player_id')

print("all players: ", len(players_country))
print("nulls in country col:", players_country['country'].isnull().sum())
print("\n uniqe country  :", players_country['country'].nunique())
print("\n all countries (to check up):")
print(sorted(players_country['country'].dropna().unique()))


KeyError: "None of [Index(['player_id', 'full_name', 'country'], dtype='str')] are in the [columns]"

In [ ]:
# win_counts رو از کدی که قبلاً برای "بیشترین برد" ساختیم داریم (winner_id, winner_name, wins)

# وصل کردن win_counts به کشور بازیکن، از طریق player_id
win_counts_with_country = win_counts.merge(players_country[['player_id', 'country']],
    left_on='winner_id',
    right_on='player_id',
    how='left')

print("wins with no country(null):", win_counts_with_country['country'].isnull().sum())

#TOTAL WIN BY COUNTRY
country_wins = win_counts_with_country.groupby('country')['wins'].sum().reset_index()
country_wins = country_wins.sort_values('wins', ascending=False)

print("\n---top ten countries ---")
print(country_wins.head(10))

top_country = country_wins.iloc[0]
print(f"\n {top_country['country']}  with total {int(top_country['wins'])} win, was the most successful country.")

wins with no country(null): 1

---top ten countries ---
           country  wins
28          France  1758
83             USA  1540
39           Italy  1369
69          Russia  1020
30         Germany   875
2        Argentina   873
76           Spain   796
42           Japan   760
3        Australia   720
21  Czech Republic   642

 France  with total 1758 win, was the most successful country.


**QUESTION SEVEN**

In [ ]:
stats_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\statistics.csv")

print("shape:", stats_df.shape)
print(stats_df['statistic_name'].unique())

shape: (1358234, 13)
<ArrowStringArray>
[                      'aces',              'double_faults',
                'first_serve',               'second_serve',
         'first_serve_points',        'second_serve_points',
       'service_games_played',         'break_points_saved',
                      'total',         'service_points_won',
        'receiver_points_won',        'max_points_in_a_row',
                  'total_won',          'service_games_won',
         'max_games_in_a_row',  'first_serve_return_points',
 'second_serve_return_points',        'return_games_played',
     'break_points_converted',                  'tiebreaks']
Length: 20, dtype: str


In [ ]:
#UNIQE PERIOD DATA
print("period uniqe:", stats_df['period'].unique())

# ONLY SEARCH IN ACES COL
aces_df = stats_df[stats_df['statistic_name'] == 'aces']
print("\n all aces:", len(aces_df))

#ACES IN EACH PERIOD
print("\n aces distribution across period rows:")
print(aces_df['period'].value_counts())

# آیا برای هر match_id، دقیقاً یه ردیف با period='ALL' داریم؟
aces_all = aces_df[aces_df['period'] == 'ALL']
print("all period=ALL:", len(aces_all))
print("uniqe  match_id  :", aces_all['match_id'].nunique())
print("duplicated match_id:", aces_all['match_id'].duplicated().sum())

period uniqe: <ArrowStringArray>
['ALL', '1ST', '2ND', '3RD']
Length: 4, dtype: str

 all aces: 76257

 aces distribution across period rows:
period
ALL    23283
1ST    23197
2ND    22823
3RD     6954
Name: count, dtype: int64
all period=ALL: 23283
uniqe  match_id  : 11389
duplicated match_id: 11894


In [ ]:
# KEEP ONLY ACES, TOTAL PER MATCH (period ALL)
df_aces = stats_df[(stats_df['statistic_name'] == 'aces') & (stats_df['period'] == 'ALL')]
df_aces = df_aces[['match_id', 'home_value', 'away_value']].drop_duplicates()

# COUNT HOW MANY TIMES EACH match_id APPEARS
match_counts = df_aces['match_id'].value_counts()

# KEEP ONLY match_ids THAT APPEAR EXACTLY ONCE (no conflict)
good_ids = match_counts[match_counts == 1].index

df_aces_clean = df_aces[df_aces['match_id'].isin(good_ids)]

print("clean aces rows:", len(df_aces_clean))
print(df_aces_clean.head())

clean aces rows: 10263
     match_id  home_value  away_value
0    11998445          12           6
71   11998446           6           3
125  11998447           8           4
178  11998448           4           0
232  11998449           6           4


In [ ]:
# TOTAL ACES PER MATCH = HOME + AWAY
df_aces_clean['total_aces'] = df_aces_clean['home_value'] + df_aces_clean['away_value']

avg_per_match = df_aces_clean['total_aces'].mean()
median_per_match = df_aces_clean['total_aces'].median()

print("--- total aces per match ---")
print(df_aces_clean['total_aces'].describe())

print(f"\nfinal answer: average aces per match = {avg_per_match:.2f}, median = {median_per_match}")

--- total aces per match ---
count    10263.000000
mean         5.351067
std          5.304342
min          0.000000
25%          2.000000
50%          4.000000
75%          8.000000
max         51.000000
Name: total_aces, dtype: float64

final answer: average aces per match = 5.35, median = 4.0


**QUESTION EIGHT**

In [ ]:
stats_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis/statistics.csv")

print("shape:", stats_df.shape)
print("\n--- data type ---")
print(stats_df.dtypes)
print("\n--- null ---")
print(stats_df.isnull().sum())

shape: (1358234, 13)

--- data type ---
match_id                     int64
period                         str
statistic_category_name        str
statistic_name                 str
home_stat                      str
away_stat                      str
compare_code                 int64
statistic_type                 str
value_type                     str
home_value                   int64
away_value                   int64
home_total                 float64
away_total                 float64
dtype: object

--- null ---
match_id                        0
period                          0
statistic_category_name         0
statistic_name                  0
home_stat                       0
away_stat                       0
compare_code                    0
statistic_type                  0
value_type                      0
home_value                      0
away_value                      0
home_total                 824739
away_total                 824739
dtype: int64


In [ ]:
# KEEP ONLY DOUBLE FAULTS, TOTAL PER MATCH (period ALL)
df_dfaults = stats_df[(stats_df['statistic_name'] == 'double_faults') & (stats_df['period'] == 'ALL')]
df_dfaults = df_dfaults[['match_id', 'home_value', 'away_value']].drop_duplicates()

# DROP MATCH_IDS WITH CONFLICTING VALUES (data errors)
match_counts = df_dfaults['match_id'].value_counts()
good_ids = match_counts[match_counts == 1].index

df_dfaults_clean = df_dfaults[df_dfaults['match_id'].isin(good_ids)]

print("clean double_faults rows:", len(df_dfaults_clean))
print(df_dfaults_clean.head())

clean double_faults rows: 10007
     match_id  home_value  away_value
1    11998445           2           7
72   11998446           2           1
126  11998447           0           5
179  11998448           2           2
233  11998449           2           3


In [ ]:


home_g = home_team.drop_duplicates().drop_duplicates(subset='match_id', keep='first')[['match_id', 'gender']].rename(columns={'gender': 'home_gender'})
away_g = away_team.drop_duplicates().drop_duplicates(subset='match_id', keep='first')[['match_id', 'gender']].rename(columns={'gender': 'away_gender'})

gender_df = home_g.merge(away_g, on='match_id', how='inner')
gender_df = gender_df.dropna(subset=['home_gender', 'away_gender'])
gender_df = gender_df[gender_df['home_gender'] == gender_df['away_gender']]

print("clean gender rows:", len(gender_df))
print(gender_df.head())

clean gender rows: 9858
   match_id home_gender away_gender
0  11998445           M           M
1  11998446           M           M
2  11998447           M           M
3  11998448           M           M
4  11998449           M           M


In [ ]:
merged = df_dfaults.merge(gender_df, on='match_id', how='inner')
print("shape after merged:", merged.shape)

shape after merged: (8059, 5)


In [ ]:
# RESHAPE TO ONE ROW PER PLAYER-MATCH
home_rows = merged[['match_id', 'home_gender', 'home_value']].rename(columns={'home_gender': 'gender', 'home_value': 'double_faults'})
away_rows = merged[['match_id', 'away_gender', 'away_value']].rename(columns={'away_gender': 'gender', 'away_value': 'double_faults'})

player_matches = pd.concat([home_rows, away_rows], ignore_index=True)
print("total player-match rows:", len(player_matches))

total player-match rows: 16118


In [ ]:
summary = player_matches.groupby('gender')['double_faults'].agg(['count', 'mean', 'median', 'std'])
print("\n--- double faults by gender ---")
print(summary)
# SIMPLE COMPARISON — JUST COMPARE THE MEANS
avg_men = player_matches.loc[player_matches['gender'] == 'M', 'double_faults'].mean()
avg_women = player_matches.loc[player_matches['gender'] == 'F', 'double_faults'].mean()

diff = avg_women - avg_men
percent_diff = (diff / avg_men) * 100

print("average double faults - men:", round(avg_men, 2))
print("average double faults - women:", round(avg_women, 2))
print("difference:", round(diff, 2))
print("percent difference:", round(percent_diff, 1), "%")
print(f"\nfinal answer: women average {avg_women:.2f} double faults vs men {avg_men:.2f} (difference = {diff:.2f}, {percent_diff:.1f}% higher)")


#print(f"\nfinal answer: women average {f_vals.mean():.2f} double faults vs men {m_vals.mean():.2f} (p-value = {p_val:.6f})")


--- double faults by gender ---
        count      mean  median       std
gender                                   
F        7046  3.531791     3.0  2.765815
M        9072  2.710979     2.0  2.226215
average double faults - men: 2.71
average double faults - women: 3.53
difference: 0.82
percent difference: 30.3 %
